In [1]:
import os
os.chdir("..")
print("Working directory:", os.getcwd()) 

Working directory: c:\Users\asdaw\Desktop\Projects\ProductReviewAI


In [2]:
import pandas as pd
import sqlite3
from src.llm.client import call_llm
from src.llm import prompts

print("Imports successful")

Imports successful


In [3]:
conn = sqlite3.connect("data/reviews.db")

# get the top reviewed product from our earlier EDA
df = pd.read_sql("""
    SELECT text, rating 
    FROM reviews_clean 
    WHERE parent_asin = 'B075X8471B' 
    LIMIT 15
""", conn)
conn.close()

print(f"Loaded {len(df)} reviews for testing")
print(df.head(3))

# combine into one text block for prompts
reviews_text = "\n".join([f"- ({row['rating']}★) {row['text']}" for _, row in df.iterrows()])
print("\n--- Combined reviews sample ---")
print(reviews_text[:500])

Loaded 15 reviews for testing
                                                text  rating
0                      Pretty cool!!! Thanks Amazon.     4.0
1                                         We love it     5.0
2  i have hemmed and hawed for quite a while abou...     5.0

--- Combined reviews sample ---
- (4.0★) Pretty cool!!! Thanks Amazon.
- (5.0★) We love it
- (5.0★) i have hemmed and hawed for quite a while about a Fire Stick or a Roku ......do i, dont i, do i dont i ........i barely watch TV and my TV in the room where i spend most of my time in has only one HDMI port .. so i finally got it on prime day for only 20 bucks and put it on my bedroom TV .. the thing is when im watching TV i fall asleep !!! but it was SIMPLE to set up and i only downloaded a few apps so far like Netflix, Youtube


In [4]:
prompt_zero = prompts.SUMMARY_ZERO_SHOT.format(reviews=reviews_text)

result_zero = call_llm(prompt_zero)

print("=== ZERO-SHOT SUMMARY ===")
print(result_zero)

=== ZERO-SHOT SUMMARY ===
Overall, customers overwhelmingly love this product, praising its ease of setup and use, especially for streaming services like Netflix and Amazon Prime. Many highlight its value in replacing expensive cable subscriptions and making any TV "smart." While a few users experienced issues with freezing, pixelation, or the voice assistant, the general sentiment is highly positive, with many recommending it for its convenience and cost-saving benefits.


In [5]:
prompt_few = prompts.SUMMARY_FEW_SHOT.format(reviews=reviews_text)

result_few = call_llm(prompt_few)

print("=== FEW-SHOT SUMMARY ===")
print(result_few)

=== FEW-SHOT SUMMARY ===
Customers overwhelmingly praise the Amazon Fire Stick for its ease of setup, intuitive interface, and ability to transform any TV into a smart TV, often highlighting its value in replacing expensive cable subscriptions. Many appreciate the voice command feature and the access to a wide range of content through apps like Netflix and Prime Video. However, some users, particularly older individuals, report frustration with the lack of traditional TV remote controls, occasional freezing or pixelation, and inconsistent performance from the voice assistant.


In [6]:
prompt_cot = prompts.SUMMARY_COT.format(reviews=reviews_text)

result_cot = call_llm(prompt_cot)

print("=== CHAIN-OF-THOUGHT SUMMARY ===")
print(result_cot)

=== CHAIN-OF-THOUGHT SUMMARY ===
The Amazon Fire Stick is largely praised for its ease of setup and use, especially for those looking to cut cable costs and access streaming services like Netflix and Prime Video. Many users appreciate its ability to make any TV "smart" and the intuitive interface, with some highlighting the effectiveness of voice commands. However, a significant minority of users experience frustrating technical issues such as freezing, slow performance, pixelation, and unreliable functionality, particularly with the Alexa voice assistant and lack of integrated TV controls, leading to strong dissatisfaction.


In [7]:
comparison = pd.DataFrame({
    "strategy": ["Zero-shot", "Few-shot", "Chain-of-Thought"],
    "output": [result_zero, result_few, result_cot],
    "length": [len(result_zero), len(result_few), len(result_cot)]
})

for _, row in comparison.iterrows():
    print(f"\n{'='*50}")
    print(f"  {row['strategy']}  (length: {row['length']} chars)")
    print(f"{'='*50}")
    print(row['output'])


  Zero-shot  (length: 450 chars)
Overall, customers overwhelmingly love this product, praising its ease of setup and use, especially for streaming services like Netflix and Amazon Prime. Many highlight its value in replacing expensive cable subscriptions and making any TV "smart." While a few users experienced issues with freezing, pixelation, or the voice assistant, the general sentiment is highly positive, with many recommending it for its convenience and cost-saving benefits.

  Few-shot  (length: 557 chars)
Customers overwhelmingly praise the Amazon Fire Stick for its ease of setup, intuitive interface, and ability to transform any TV into a smart TV, often highlighting its value in replacing expensive cable subscriptions. Many appreciate the voice command feature and the access to a wide range of content through apps like Netflix and Prime Video. However, some users, particularly older individuals, report frustration with the lack of traditional TV remote controls, occasional fre

Prompt Strategy Comparison — Product Summary Task:

Zero-shot (450 chars):
  Best for: quick overviews where speed matters
  Weakness: generic, misses product-specific details

Few-shot (557 chars):
  Best for: structured business summaries
  Strength: examples prime the model to be specific
  Chosen as default for product summary module

Chain-of-Thought (599 chars):
  Best for: detailed analysis, executive reports
  Strength: captures nuance and minority complaints
  Chosen for executive report module

Key insight: prompt strategy should match the use case.
Few-shot works best when output format matters.
CoT works best when completeness matters.
Zero-shot works best when speed matters.

In [9]:
import sqlite3

results_df = pd.DataFrame({
    "task": ["summary", "summary", "summary"],
    "strategy": ["zero_shot", "few_shot", "cot"],
    "prompt": [
        prompts.SUMMARY_ZERO_SHOT,
        prompts.SUMMARY_FEW_SHOT,
        prompts.SUMMARY_COT
    ],
    "output": [result_zero, result_few, result_cot],
    "output_length": [len(result_zero), len(result_few), len(result_cot)]
})

conn = sqlite3.connect("data/reviews.db")
results_df.to_sql("prompt_experiments", conn, if_exists="replace", index=False)
conn.close()

print("Saved prompt experiments → SQLite")
print(results_df[["task", "strategy", "output_length"]])

Saved prompt experiments → SQLite
      task   strategy  output_length
0  summary  zero_shot            450
1  summary   few_shot            557
2  summary        cot            599
